# Data Preparation & Feature Engineering 
This notebook converts the raw MEPS HC-252 longitudinal file into two datasets:

1) **df_pre (core, cleaned dataset)**  
   A reduced set of “core” variables (whitelist) with basic cleaning applied (special missing codes → NaN, negative expenditures → NaN, and a consistent longitudinal sample filter).

2) **df_feat (feature table)**  
   Adds engineered predictors (Year 1) and constructs Year 2 outcomes/labels for modeling:
   - Regression: `LOG_TOTEXPY2`
   - Classification: `HIGHCOST_Y2`, `ANY_ED_Y2`, `ANY_IP_Y2`

Finally, the notebook defines a **scikit-learn preprocessing pipeline** (imputation + one-hot encoding) to support baseline models.



## 0) Setup and imports

In [20]:


from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Make project importable (so we can `import src.*`) ---
PROJECT_ROOT = Path.cwd().resolve().parents[0]  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# --- Project constants + helpers ---
from src.config import (
    RAW_DATA_PATH,
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    REG_BASELINE_TOTEXPY1,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
)

from src.data_io import load_raw_meps, save_processed_meps
from src.preprocessing import preprocess_meps
from src.features import add_features


PROJECT_ROOT: /Users/wenxi/Desktop/TFM_25


## 1) Load raw MEPS data

In [21]:

df_raw = load_raw_meps()
print("df_raw:", df_raw.shape)

df_raw: (8292, 2648)


## 2) Preprocess: build the core cleaned dataset (`df_pre`)

**Goal:** Reduce the 2,600+ raw columns to a manageable “core” dataset and apply lightweight cleaning.

`preprocess_meps(df_raw)` performs:
- **Whitelist selection:** keep only predefined core variables (IDs/weights, demographics, SES, insurance, employment summaries, health status, chronic indicators, utilization/cost).
- **Longitudinal sample filter:** restrict to complete panel participants where available:
  - `ALL5RDS == 1` (responded in all rounds)
  - `YEARIND == 1` (present in both years)
- **Missing codes → NaN:** recode MEPS special negative missing codes to standard missing values.
- **Sanitize expenditures:** set negative dollar amounts to NaN.
- **Drop near-empty columns:** remove coverage flags like `PREVCOVR` / `MORECOVR` if they are almost entirely missing.


In [22]:

df_pre = preprocess_meps(df_raw)
print("df_pre:", df_pre.shape)

df_pre: (7812, 101)


## 3) Save the preprocessed dataset (`df_pre`)

In [4]:
from src.config import PROCESSED_DATA_PATH
from src.data_io import save_processed_meps


save_processed_meps(df_pre, PROCESSED_DATA_PATH)  
print("Saved preprocessed dataset to:", PROCESSED_DATA_PATH)


Saved preprocessed dataset to: /Users/wenxi/Desktop/TFM_25/data/meps_panel27_processed.parquet



## 4) Feature engineering: create `df_feat`

**Goal:** Add engineered predictors and construct modeling targets/labels.

- Run `df_feat = add_features(df_pre)`.
- This step typically:
  - Creates interpretable demographic/SES transformations (e.g., age groups, poverty labels).
  - Creates insurance type labels (for one-hot encoding).
  - Builds employment attachment summaries across rounds.
  - Builds health/chronic indicators and multi-morbidity counts.
  - Constructs baseline utilization features (Year 1) and Year 2 targets.

In [5]:
# feature engineering pipline

df_feat = add_features(df_pre)

df_feat.head()


,DUID,PID,DUPERSID,PANEL,YEARIND,ALL5RDS,DIED,INST,MILITARY,ENTRSRVY,...,DIABDXY1_M18_BIN,MULTIMORBIDITY_Y1,MULTIMORBIDITY_GE2,LOG_TOTEXPY1,ANY_ED_Y1,ANY_IP_Y1,LOG_TOTEXPY2,HIGHCOST_Y2,ANY_ED_Y2,ANY_IP_Y2
0,2790002,101,2790002101,27,1,1,0,0,0,0,...,1,1,0,7.595890,0,0,6.472346,0,0,0
1,2790002,102,2790002102,27,1,1,0,0,0,0,...,0,1,0,0.000000,0,0,7.546974,0,0,0
2,2790004,101,2790004101,27,1,1,0,0,0,0,...,0,0,0,7.379632,0,0,6.894670,0,0,0
3,2790006,101,2790006101,27,1,1,0,0,0,0,...,1,3,1,7.374629,0,0,7.180070,0,0,0
4,2790006,102,2790006102,27,1,1,0,0,0,0,...,0,0,0,5.017280,0,0,0.000000,0,0,0


## 5) Save the feature dataset

In [6]:
from pathlib import Path
from src.config import PROJECT_ROOT

out_path = PROJECT_ROOT / "data" / "df_feat.parquet"
df_feat.to_parquet(out_path, index=False)
print("Saved to:", out_path)


Saved to: /Users/wenxi/Desktop/TFM_25/data/df_feat.parquet


## 6) Sanity check: confirm targets exist

**Goal:** Verify that the expected outcome columns were created successfully.

- Check that these columns exist in `df_feat`:
  - `TOTEXPY2`, `LOG_TOTEXPY2`
  - `HIGHCOST_Y2`, `ANY_ED_Y2`, `ANY_IP_Y2`
- Summarize these columns using `.describe()`.

In [23]:
from src.config import (
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
)

cols = [
    REG_TARGET_TOTEXPY2_RAW,
    REG_TARGET_TOTEXPY2_LOG,
    CLASS_TARGET_HIGHCOST_Y2,
    CLASS_TARGET_ANY_ED_Y2,
    CLASS_TARGET_ANY_IP_Y2,
]
[c for c in cols if c in df_feat.columns]


['TOTEXPY2', 'LOG_TOTEXPY2', 'HIGHCOST_Y2', 'ANY_ED_Y2', 'ANY_IP_Y2']

In [8]:
df_feat[[c for c in cols if c in df_feat.columns]].describe(include="all")


,TOTEXPY2,LOG_TOTEXPY2,HIGHCOST_Y2,ANY_ED_Y2,ANY_IP_Y2
count,7812.000000,7812.000000,7812.000000,7812.000000,7812.000000
mean,8304.398874,6.669023,0.100102,0.142729,0.072581
std,22264.311550,3.227354,0.300156,0.349819,0.259464
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,272.750000,5.612214,0.000000,0.000000,0.000000
50%,1815.500000,7.504667,0.000000,0.000000,0.000000
75%,7268.250000,8.891408,0.000000,0.000000,0.000000
max,574675.000000,13.261562,1.000000,1.000000,1.000000


## 7) Check label prevalence and class imbalance

**Goal:** Inspect the distribution of binary targets before modeling.

- Print `value_counts()` for:
  - `HIGHCOST_Y2`
  - `ANY_ED_Y2`
  - `ANY_IP_Y2`

This confirms how imbalanced each classification task is and informs later choices (e.g., class weights, PR-AUC, thresholding, etc.).


In [24]:
targets = [CLASS_TARGET_HIGHCOST_Y2, CLASS_TARGET_ANY_ED_Y2, CLASS_TARGET_ANY_IP_Y2]
targets = [c for c in targets if c in df_feat.columns]

rows = []
for c in targets:
    vc = df_feat[c].value_counts(dropna=False)
    n_total = len(df_feat)
    n0 = int(vc.get(0, 0))
    n1 = int(vc.get(1, 0))
    n_na = int(vc.get(np.nan, 0))  # might be 0 if none

    denom = n0 + n1  # exclude NaN from rate
    pos_rate = (n1 / denom * 100) if denom > 0 else np.nan

    rows.append({
        "target": c,
        "n_total": n_total,
        "n_1": n1,
        "n_0": n0,
        "n_missing": n_na,
        "pos_rate_%": round(pos_rate, 2),
    })

summary = pd.DataFrame(rows).sort_values("pos_rate_%", ascending=False)
summary


,target,n_total,n_1,n_0,n_missing,pos_rate_%
1,ANY_ED_Y2,7812,1115,6697,0,14.27
0,HIGHCOST_Y2,7812,782,7030,0,10.01
2,ANY_IP_Y2,7812,567,7245,0,7.26


Overall, these outcomes represent **imbalanced classification problems**, especially for inpatient events (ANY_IP_Y2). This motivates using imbalance-aware evaluation (e.g., PR-AUC in addition to ROC AUC) and potentially applying class weighting or threshold calibration during model training.

## 8) Define candidate predictors (exclude IDs/weights/targets)

**Goal:** Build a clean set of candidate features for modeling.

- Exclude columns that must not be used as predictors:
  - IDs (e.g., `DUPERSID`, `DUID`, `PID`, `PANEL`)
  - Survey design variables (e.g., `VARSTR`, `VARPSU`) and weights (e.g., `LONGWT`, `LSAQWT`)
  - Targets (e.g., `TOTEXPY2`, `LOG_TOTEXPY2`, `HIGHCOST_Y2`, `ANY_ED_Y2`, `ANY_IP_Y2`)
- From the remaining columns, split into:
  - **Categorical columns** (`object` / `category`)
  - **Numeric columns** (int/float/bool)

In [10]:
import numpy as np
import pandas as pd

# 1) define columns we NEVER want as model predictors
ID_COLS = ["DUPERSID", "DUID", "PID", "PANEL", "VARSTR", "VARPSU"]
WEIGHT_COLS = ["LONGWT", "LSAQWT"]

# targets 
TARGET_COLS = ["TOTEXPY2", "LOG_TOTEXPY2", "HIGHCOST_Y2", "ANY_ED_Y2", "ANY_IP_Y2"]

EXCLUDE = set(ID_COLS + WEIGHT_COLS + TARGET_COLS)

# 2) candidate feature columns = everything else
feature_candidates = [c for c in df_feat.columns if c not in EXCLUDE]

# 3) categorical = object/category
cat_cols = df_feat[feature_candidates].select_dtypes(include=["object", "category"]).columns.tolist()

# 4) numeric = number types (int/float/bool)
num_cols = df_feat[feature_candidates].select_dtypes(include=[np.number]).columns.tolist()

print("n feature candidates:", len(feature_candidates))
print("n cat:", len(cat_cols))
print("n num:", len(num_cols))

cat_cols[:20], num_cols[:20]


n feature candidates: 128
n cat: 7
n num: 121


(['AGE_GROUP',
  'RACE_ETH',
  'REGIONY1_CAT',
  'EDU_GROUP',
  'POVCATY1_CAT',
  'FAMSIZE_Y1_GRP',
  'INS_TYPE_Y1'],
 ['YEARIND',
  'ALL5RDS',
  'DIED',
  'INST',
  'MILITARY',
  'ENTRSRVY',
  'LEFTUS',
  'OTHER',
  'AGEY1X',
  'AGEY2X',
  'AGELSTY1',
  'AGELSTY2',
  'SEX',
  'RACETHX',
  'HISPANX',
  'EDUCYR',
  'REGIONY1',
  'REGIONY2',
  'FAMINCY1',
  'FAMINCY2'])

## 9) Choose categorical features (manual shortlist)

**Goal:** Use a controlled, interpretable set of categorical predictors for baseline models.

- Define `cat_cols` explicitly (e.g., race/ethnicity, region, education group, poverty category, family size group, insurance type).
- This avoids accidental inclusion of unwanted or redundant columns.

**Modeling note:**  
Use `AGE` (continuous) for modeling. Keep `AGE_GROUP` mainly for plots/interpretation.


In [12]:
cat_cols = [
    "RACE_ETH",
    "REGIONY1_CAT",
    "EDU_GROUP",
    "POVCATY1_CAT",
    "FAMSIZE_Y1_GRP",
    "INS_TYPE_Y1",
]


## 10) Quick QA checks for engineered employment features

**Goal:** Ensure the employment summary variables behave as intended, especially around missing/NIU.

- Inspect `EMP_INFO_R12` (whether any round-level employment info exists).
- Inspect missingness and distribution of `EMP_ATTACHED_ANY_R12`.
- Confirm that `EMP_ATTACHED_ANY_R12_FILL0` is model-friendly (no NaN), while `EMP_INFO_R12` preserves the “no information” signal.

In [13]:
df_feat["EMP_INFO_R12"].value_counts(dropna=False)


EMP_INFO_R12
1    6492
0    1320
Name: count, dtype: int64

In [14]:
df_feat["EMP_ATTACHED_ANY_R12"].isna().mean()
df_feat["EMP_ATTACHED_ANY_R12"].value_counts(dropna=False)


EMP_ATTACHED_ANY_R12
1.0    3979
0.0    2513
NaN    1320
Name: count, dtype: int64

In [15]:
df_feat["EMP_ATTACHED_ANY_R12_FILL0"].isna().mean()

np.float64(0.0)

## 11) Choose numeric features (manual shortlist)

**Goal:** Define a stable numeric feature set using engineered columns (not raw codes).

Typical numeric blocks include:
- Demographics/SES: `AGE`, `SEX_BIN`, `LOG_FAMINCY1`, `FAMSIZE_Y1`
- Employment: `WORKED_Y1`, unemployment compensation features, employment info flags
- Health status: fair/poor indicators
- Chronic conditions: *_BIN indicators (and optionally a multi-morbidity count)
- Baseline utilization/cost: `LOG_TOTEXPY1`, `ANY_ED_Y1`, `ANY_IP_Y1`


In [ ]:
#Numeric (use engineered columns, not raw)

num_cols = [
    # demographics / SES
    "AGE",
    "SEX_BIN",
    "LOG_FAMINCY1",
    "FAMSIZE_Y1",

    # insurance 
    # we decide to keep INS_TYPE_Y1, and drop these three to avoid redundancy:
    # "ANY_PRIVATE_Y1", "PUBLIC_ONLY_Y1", "UNINSURED_Y1",

    # employment
    "WORKED_Y1",
    "ANY_UNEMP_COMP_Y1",
    "LOG_UNEMP_COMP_Y1",
    "EMP_INFO_R12",
    "EMP_ATTACHED_ANY_R12_FILL0",  # model-friendly version

    # health status baseline
    "RTHLTH1_FAIRPOOR",
    "MNHLTH1_FAIRPOOR",

    # chronic conditions baseline
    "HIBPDXY1_BIN",
    "CHDDXY1_BIN",
    "STRKDXY1_BIN",
    "CHOLDXY1_BIN",
    "ASTHDXY1_BIN",
    "DIABDXY1_M18_BIN",
    # "MULTIMORBIDITY_Y1",   # optional (can remove if you keep all *_BIN)

    # baseline utilisation/cost
    "LOG_TOTEXPY1",
    "ANY_ED_Y1",
    "ANY_IP_Y1",
]


## 12) Build the scikit-learn preprocessing pipeline

**Goal:** Create a reusable transformer that converts raw feature columns into a model-ready matrix.

- Numeric pipeline:
  - median imputation (`SimpleImputer(strategy="median")`)
- Categorical pipeline:
  - most-frequent imputation (`SimpleImputer(strategy="most_frequent")`)
  - one-hot encoding (`OneHotEncoder(handle_unknown="ignore")`)
- Use `ColumnTransformer` to apply these pipelines to `num_cols` and `cat_cols`.
- Drop all other columns (`remainder="drop"`).

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="drop",
)
